In [ ]:
# ==========================================================
# SECTION 1: ENVIRONMENT SETUP & IMPORTS
# ==========================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import logging
from datetime import datetime

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Optional: XGBoost for industry-grade performance
try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed. Install via: pip install xgboost")

# Logging configuration (production-style)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Plot styling
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

logger.info("Environment setup complete.")

In [ ]:
# ==========================================================
# SECTION 2: DATA INGESTION
# ==========================================================
FILE_PATH = r"C:\Users\vivek\Downloads\Global Superstore.csv"

def load_data(path):
    """Load raw retail transaction data with encoding handling."""
    try:
        df = pd.read_csv(path, encoding='latin1')
        logger.info(f"Data loaded successfully. Shape: {df.shape}")
        return df
    except FileNotFoundError:
        logger.error(f"File not found at {path}")
        raise
    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        raise

df = load_data(FILE_PATH)
print("Columns available:", df.columns.tolist())
df.head()

In [ ]:
# ==========================================================
# SECTION 3: DATA QUALITY AUDIT
# ==========================================================
def audit_data(df):
    """Perform a data quality audit before modeling."""
    audit = pd.DataFrame({
        'dtype': df.dtypes,
        'missing': df.isnull().sum(),
        'missing_pct': (df.isnull().sum() / len(df)) * 100,
        'unique': df.nunique()
    })
    print("=== DATA QUALITY REPORT ===")
    print(audit)
    print(f"\nDuplicate rows: {df.duplicated().sum()}")
    return audit

audit_data(df)

In [ ]:
# ==========================================================
# SECTION 4: FEATURE ENGINEERING (Temporal + Business)
# ==========================================================
def engineer_features(df):
    """Create industry-grade temporal and business features."""
    df = df.copy()
    
    # --- Date parsing ---
    df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True, errors='coerce')
    df = df.dropna(subset=['Order Date', 'Sales'])
    df = df.sort_values('Order Date').reset_index(drop=True)
    
    # --- Calendar features ---
    df['Year'] = df['Order Date'].dt.year
    df['Month'] = df['Order Date'].dt.month
    df['Quarter'] = df['Order Date'].dt.quarter
    df['Week'] = df['Order Date'].dt.isocalendar().week.astype(int)
    df['DayOfWeek'] = df['Order Date'].dt.dayofweek
    df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
    df['DayOfYear'] = df['Order Date'].dt.dayofyear
    
    # --- Cyclical encoding (critical for ML on seasonality) ---
    df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
    df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)
    df['Dow_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
    df['Dow_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)
    
    # --- Business features ---
    if 'Discount' in df.columns:
        df['HasDiscount'] = (df['Discount'] > 0).astype(int)
    if 'Quantity' in df.columns and 'Sales' in df.columns:
        df['UnitPrice'] = df['Sales'] / df['Quantity'].replace(0, np.nan)
        df['UnitPrice'].fillna(df['UnitPrice'].median(), inplace=True)
    
    # --- Lag features (previous sales patterns) ---
    daily = df.groupby('Order Date')['Sales'].sum().reset_index()
    daily['Lag_1'] = daily['Sales'].shift(1)
    daily['Lag_7'] = daily['Sales'].shift(7)
    daily['Lag_30'] = daily['Sales'].shift(30)
    daily['Rolling_Mean_7'] = daily['Sales'].rolling(window=7).mean()
    daily['Rolling_Mean_30'] = daily['Sales'].rolling(window=30).mean()
    daily['Rolling_STD_7'] = daily['Sales'].rolling(window=7).std()
    
    return df, daily

df, daily_sales = engineer_features(df)
logger.info(f"Feature engineering complete. Final shape: {df.shape}")
df.head()

In [ ]:
# ==========================================================
# SECTION 5: EXPLORATORY DATA ANALYSIS
# ==========================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Overall sales trend
daily_sales.plot(x='Order Date', y='Sales', ax=axes[0,0], color='steelblue', legend=False)
axes[0,0].set_title('Daily Sales Trend Over Time')
axes[0,0].set_xlabel('Date'); axes[0,0].set_ylabel('Sales')

# 2. Monthly seasonality
monthly = df.groupby('Month')['Sales'].sum()
monthly.plot(kind='bar', ax=axes[0,1], color='coral')
axes[0,1].set_title('Total Sales by Month (Seasonality)')
axes[0,1].set_xlabel('Month'); axes[0,1].set_ylabel('Total Sales')

# 3. Sales by Category
if 'Category' in df.columns:
    cat_sales = df.groupby('Category')['Sales'].sum().sort_values()
    cat_sales.plot(kind='barh', ax=axes[1,0], color='seagreen')
    axes[1,0].set_title('Sales by Product Category')
    axes[1,0].set_xlabel('Total Sales')

# 4. Day of week pattern
dow = df.groupby('DayOfWeek')['Sales'].mean()
dow.plot(kind='bar', ax=axes[1,1], color='purple')
axes[1,1].set_title('Average Sales by Day of Week')
axes[1,1].set_xlabel('Day (0=Mon, 6=Sun)'); axes[1,1].set_ylabel('Avg Sales')

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# SECTION 6: PREPROCESSING PIPELINE
# ==========================================================
def preprocess(df):
    """Encode categoricals and prepare features for modeling."""
    df = df.copy()
    
    drop_cols = ['Row ID', 'Order ID', 'Customer ID', 'Customer Name',
                 'Product ID', 'Product Name', 'Order Date', 'Ship Date',
                 'Postal Code', 'Country', 'City', 'State']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    
    # Encode categorical variables
    le_dict = {}
    for col in df.select_dtypes(include='object').columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        le_dict[col] = le
    
    return df, le_dict

df_model, encoders = preprocess(df)
logger.info(f"Preprocessing done. Modeling features: {df_model.shape[1] - 1}")

In [ ]:
# ==========================================================
# SECTION 7: MODEL TRAINING & COMPARISON
# ==========================================================
X = df_model.drop('Sales', axis=1)
y = df_model['Sales']

# TimeSeriesSplit for realistic temporal validation
tscv = TimeSeriesSplit(n_splits=5)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
}

if XGB_AVAILABLE:
    models['XGBoost'] = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42)

results = []
trained_models = {}

for name, model in models.items():
    logger.info(f"Training {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    mape = np.mean(np.abs((y_test - preds) / np.where(y_test == 0, 1, y_test))) * 100
    
    results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE (%)': mape})
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values('RMSE')
print("\n=== MODEL COMPARISON ===\n")
print(results_df.to_string(index=False))

In [ ]:
# ==========================================================
# SECTION 8: HYPERPARAMETER TUNING (Best Model)
# ==========================================================
best_model_name = results_df.iloc[0]['Model']
logger.info(f"Tuning best model: {best_model_name}")

if best_model_name in ['Random Forest', 'XGBoost']:
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20, None],
        'min_samples_split': [2, 5]
    }
    base = RandomForestRegressor(random_state=42, n_jobs=-1) if best_model_name == 'Random Forest' else XGBRegressor(random_state=42)
    
    grid = GridSearchCV(base, param_grid, cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1)
    grid.fit(X_train, y_train)
    
    final_model = grid.best_estimator_
    logger.info(f"Best params: {grid.best_params_}")
else:
    final_model = trained_models[best_model_name]

# Final predictions
final_preds = final_model.predict(X_test)
final_r2 = r2_score(y_test, final_preds)
final_rmse = np.sqrt(mean_squared_error(y_test, final_preds))
print(f"\nFINAL MODEL: {best_model_name}")
print(f"R2: {final_r2:.4f} | RMSE: {final_rmse:.2f}")

In [ ]:
# ==========================================================
# SECTION 9: FEATURE IMPORTANCE
# ==========================================================
if hasattr(final_model, 'feature_importances_'):
    importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': final_model.feature_importances_
    }).sort_values('Importance', ascending=False).head(15)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Feature', data=importance, palette='viridis')
    plt.title(f'Top 15 Feature Importances - {best_model_name}')
    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================================
# SECTION 10: RESIDUAL ANALYSIS
# ==========================================================
residuals = y_test - final_preds

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].scatter(final_preds, residuals, alpha=0.4, color='teal')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted Sales'); axes[0].set_ylabel('Residuals')
axes[0].set_title('Residual Plot')

axes[1].hist(residuals, bins=40, color='orange', edgecolor='black')
axes[1].set_xlabel('Residual'); axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')

plt.tight_layout(); plt.show()

In [ ]:
# ==========================================================
# SECTION 11: MODEL PERSISTENCE
# ==========================================================
joblib.dump(final_model, 'retail_sales_forecast_model.pkl')
joblib.dump(encoders, 'label_encoders.pkl')
logger.info("Model and encoders saved successfully.")
print("Artifacts saved: retail_sales_forecast_model.pkl, label_encoders.pkl")